# Decision Tree and Random Forest

My part of the project was to implement and test the decision tree and random forest using Python and NumPy. I used my teammate's saved training and validation splits so we could compare the models on the same rows.

This notebook contains my model code, my recorded results, and explanations of what I found. The tables come from our earlier terminal runs. Set `RUN_EXPERIMENTS = True` to run those experiments again. Set `RUN_ERROR_ANALYSIS = True` to calculate the error analysis. These sections take several minutes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from comp472_project.config import SEED, SENTIMENT_LABELS
from comp472_project.data import load_split
from comp472_project.evaluation import evaluate, confusion_matrix, per_class_accuracy, most_confused_pairs

RUN_EXPERIMENTS = False
RUN_ERROR_ANALYSIS = False

## How my models work

The decision tree asks questions about the features, such as whether a value is below a threshold. In `fit`, I sort the rows by each candidate feature and test cuts between different values. I count the labels on each side and keep the cut that reduces impurity the most. The tree repeats this process until it reaches a stopping rule. Each leaf stores its most common label. In `predict`, each row follows the questions until it reaches a leaf.

`max_depth` limits the number of levels. `min_samples_split` sets how many rows a node needs before trying a split. `min_samples_leaf` sets how many rows each child must keep. `criterion` chooses Gini or entropy.

The random forest trains several decision trees. Each tree gets a random sample of the training rows and considers a random set of features at each split. Every tree makes a prediction, and the most common prediction becomes the forest's answer.

## My code

The next two cells contain my decision tree from the attached file and my random forest code. The forest imports `DecisionTreeClassifier` from the project package, as in my original file. I checked that the imported tree has the same implementation as the attached tree.

In [ ]:
"""Decision Tree classifier, implemented from scratch with NumPy."""

import numpy as np


class DecisionTreeClassifier:
    """
    Parameters
    ----------
    max_depth : maximum depth of the tree (None = grow until pure or other
        stopping rules apply).
    min_samples_split : minimum samples a node needs to be considered for a split.
    min_samples_leaf : minimum samples each child must keep after a split.
    criterion : "gini" or "entropy".
    max_features : features considered at each split: None (all), "sqrt",
        "log2", an int, or a float fraction. Random Forest uses this.
    seed : seed for the feature subsampling.
    """

    def __init__(
        self,
        max_depth: int | None = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        criterion: str = "gini",
        max_features: int | float | str | None = None,
        seed: int | None = None,
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.max_features = max_features
        self.seed = seed

    def fit(self, X: np.ndarray, y: np.ndarray) -> "DecisionTreeClassifier":
        # TODO: recursively choose the (feature, threshold) split that most
        # reduces impurity, until a stopping rule is met.
        X = np.asarray(X, dtype=float)
        classes, labels = np.unique(y, return_inverse=True)
        n_rows, n_features = X.shape

        if n_rows == 0 or n_features == 0:
            raise ValueError("X must have at least one row and one feature")
        if self.criterion not in ("gini", "entropy"):
            raise ValueError("criterion must be 'gini' or 'entropy'")

        setting = self.max_features
        if setting is None:
            n_choices = n_features
        elif setting == "sqrt":
            n_choices = max(1, int(np.sqrt(n_features)))
        elif setting == "log2":
            n_choices = max(1, int(np.log2(n_features)))
        elif isinstance(setting, (int, np.integer)):
            n_choices = setting
        elif isinstance(setting, (float, np.floating)) and 0 < setting <= 1:
            n_choices = max(1, int(setting * n_features))
        else:
            raise ValueError("Invalid max_features setting")

        if not 1 <= n_choices <= n_features:
            raise ValueError("max_features must select between 1 and all features")

        rng = np.random.default_rng(self.seed)
        one_hot = np.eye(len(classes), dtype=np.int32)[labels]

        def impurity(counts):
            probs = counts / counts.sum(axis=-1, keepdims=True)
            if self.criterion == "gini":
                return 1 - (probs ** 2).sum(axis=-1)
            log_probs = np.zeros_like(probs)
            np.log2(probs, out=log_probs, where=probs > 0)
            return -(probs * log_probs).sum(axis=-1)

        def grow(rows, depth):
            size = len(rows)
            counts = one_hot[rows].sum(axis=0)
            majority = int(classes[counts.argmax()])
            if (
                np.count_nonzero(counts) == 1
                or (self.max_depth is not None and depth >= self.max_depth)
                or size < self.min_samples_split
                or size < 2 * self.min_samples_leaf
            ):
                return majority
            features = range(n_features)
            if n_choices < n_features:
                features = rng.choice(n_features, size=n_choices, replace=False)

            best_score = impurity(counts) - 1e-12
            best_split = None
            for feature in features:
                sorted_rows = rows[np.argsort(X[rows, feature])]
                values = X[sorted_rows, feature]
                # only cut between different values
                cuts = np.flatnonzero(values[:-1] < values[1:]) + 1
                cuts = cuts[
                    (cuts >= self.min_samples_leaf)
                    & (size - cuts >= self.min_samples_leaf)
                ]
                if cuts.size == 0:
                    continue

                # count the labels on each side of each cut
                running_counts = np.cumsum(one_hot[sorted_rows], axis=0, dtype=np.int32)
                left_counts = running_counts[cuts - 1]
                right_counts = counts - left_counts
                scores = (
                    cuts * impurity(left_counts)
                    + (size - cuts) * impurity(right_counts)
                ) / size
                best = scores.argmin()
                if scores[best] < best_score:
                    best_score = scores[best]
                    cut = cuts[best]
                    lower, upper = values[cut - 1], values[cut]
                    threshold = lower / 2 + upper / 2
                    if threshold >= upper:
                        threshold = lower
                    best_split = (feature, threshold, sorted_rows, cut)
            if best_split is None:
                return majority
            feature, threshold, sorted_rows, cut = best_split
            left = grow(sorted_rows[:cut], depth + 1)
            right = grow(sorted_rows[cut:], depth + 1)
            return feature, threshold, left, right
        self._tree = grow(np.arange(n_rows), depth=0)
        return self
        
    def predict(self, X: np.ndarray) -> np.ndarray:
        # TODO: route each row from the root to a leaf; return the leaf's class.
        if not hasattr(self, "_tree"):
            raise ValueError("Call fit before predict")
        X = np.asarray(X, dtype=np.float64)
        predictions = np.empty(len(X), dtype=int)
        for i, row in enumerate(X):
            node = self._tree
            while isinstance(node, tuple):
                feature, threshold, left, right = node
                node = left if row[feature] <= threshold else right
            predictions[i] = node
        return predictions


In [ ]:
"""Random Forest, built from the from-scratch DecisionTreeClassifier."""

import numpy as np

from comp472_project.models.decision_tree import DecisionTreeClassifier


class RandomForestClassifier:
    """
    Parameters
    ----------
    n_estimators : number of trees.
    max_depth, min_samples_split, min_samples_leaf, criterion : passed to each tree.
    max_features : features considered at each split in each tree.
    bootstrap : if True, each tree trains on a bootstrap sample of the rows.
    seed : seed for bootstrap sampling and per-tree seeds.
    """

    def __init__(
        self,
        n_estimators: int = 100,
        max_depth: int | None = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        criterion: str = "gini",
        max_features: int | float | str | None = "sqrt",
        bootstrap: bool = True,
        seed: int | None = None,
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.seed = seed

    def fit(self, X: np.ndarray, y: np.ndarray) -> "RandomForestClassifier":
        # TODO: train n_estimators DecisionTreeClassifier instances, each on a
        # bootstrap sample and with its own seed.
        if self.n_estimators < 1:
            raise ValueError("n_estimators must be at least 1")
        X = np.asarray(X)
        y = np.asarray(y)
        rng = np.random.default_rng(self.seed)
        trees = []
        for _ in range(self.n_estimators):
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                criterion=self.criterion,
                max_features=self.max_features,
                seed=int(rng.integers(0, 2**32)),
            )

            if self.bootstrap:
                # Pick training rows with replacement for this tree.
                rows = rng.integers(0, len(y), size=len(y))
                tree.fit(X[rows], y[rows])
            else:
                tree.fit(X, y)
            trees.append(tree)
        self.trees_ = trees
        self.classes_ = np.unique(y)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        # TODO: majority vote across trees.
        if not hasattr(self, "trees_"):
            raise ValueError("Call fit before predict")
        votes = np.zeros((len(X), len(self.classes_)), dtype=int)
        rows = np.arange(len(X))
        for tree in self.trees_:
            labels = tree.predict(X)
            # Find the vote column for each predicted label.
            columns = np.searchsorted(self.classes_, labels)
            votes[rows, columns] += 1
        return self.classes_[votes.argmax(axis=1)]


In `fit`, I sample row numbers with replacement, so one row can appear more than once in a tree's training sample. I give each tree its own seed. This makes the trees different while letting me repeat the same experiment with seed 42.

The forest passes `max_features` to the tree. With `"sqrt"`, each split considers 22 features for images or 27 for sentiment. With `bootstrap=False`, each tree uses all training rows.

In `predict`, I count the trees' votes for every class. `searchsorted` finds the column for each predicted label, and `argmax` picks the class with the most votes. A tie goes to the smallest label.

## Training and validation

I used the existing 80/20 splits in `data/splits/`. The models learn from the training rows. I then check their predictions on the validation rows, which they did not train on. The IDs are used to identify rows and are excluded from the features. I used the supplied features directly and seed 42 for these experiments.

I also ran the existing tree and forest tests. Both passed in 14.88 seconds in the original terminal run.

In [ ]:
data = {task: load_split(task) for task in ("image", "sentiment")}
summary = []
for task, split in data.items():
    summary.append({"task": task, "training_rows": len(split.y_train),
                    "validation_rows": len(split.y_val), "features": split.X_train.shape[1],
                    "classes": split.n_classes})
display(pd.DataFrame(summary))

## Check with my training data

This cell fits both models on the first 100 rows of the saved sentiment training split. It checks the prediction shape and label range. The printed accuracy is on these same training rows, so it is only a quick implementation check. The later experiments use the full training and validation splits.

In [ ]:
sample = data["sentiment"]
X = sample.X_train[:100]
y = sample.y_train[:100]
for model in (DecisionTreeClassifier(max_depth=2, seed=SEED),
              RandomForestClassifier(n_estimators=10, max_depth=2, seed=SEED)):
    model.fit(X, y)
    pred = model.predict(X)
    assert pred.shape == y.shape
    assert set(np.unique(pred)).issubset(set(np.unique(y)))
    print(type(model).__name__, "training subset accuracy:", round(np.mean(pred == y), 3))

## Recorded experiments

Accuracies below are percentages copied from the earlier terminal output. Missing fit times were not recorded. The 60-tree validation measurements were run with AI assistance and reported in the chat. All runs used seed 42. Forests used bootstrap sampling. Each row lists its complete model settings.

In [ ]:
rows = [
    ("image", "tree", 0, 10, 1, 2, "gini", "all", 9.2, 7.2, np.nan),
    ("image", "tree", 0, 20, 1, 2, "gini", "all", 19.5, 9.8, 41.7),
    ("image", "tree", 0, 20, 1, 20, "gini", "all", 16.6, 9.9, 38.4),
    ("image", "tree", 0, 20, 1, 2, "entropy", "all", 67.8, 9.7, 81.1),
    ("image", "forest", 10, 10, 1, 2, "gini", "sqrt", 15.2, 10.3, np.nan),
    ("image", "forest", 30, 20, 1, 2, "gini", "sqrt", 48.7, 15.3, np.nan),
    ("image", "forest", 30, 20, 1, 2, "gini", "0.1", 34.6, 14.5, 119.0),
    ("image", "forest", 60, 20, 1, 2, "gini", "sqrt", 55.7875, 16.65, np.nan),
    ("sentiment", "tree", 0, 10, 1, 2, "gini", "all", 82.1, 46.9, np.nan),
    ("sentiment", "tree", 0, 10, 5, 2, "gini", "all", 79.3, 47.2, 16.2),
    ("sentiment", "tree", 0, 10, 5, 20, "gini", "all", 76.9, 46.9, 14.7),
    ("sentiment", "tree", 0, 10, 5, 2, "entropy", "all", 77.8, 45.5, 20.4),
    ("sentiment", "forest", 10, 10, 1, 2, "gini", "sqrt", 85.1, 52.8, np.nan),
    ("sentiment", "forest", 30, 10, 5, 2, "gini", "sqrt", 91.2, 57.2, np.nan),
    ("sentiment", "forest", 30, 10, 5, 2, "gini", "0.1", 92.9, 56.5, 42.4),
    ("sentiment", "forest", 60, 10, 5, 2, "gini", "sqrt", 93.4375, 57.2916667, np.nan),
]
results = pd.DataFrame(rows, columns=["task", "model", "trees", "depth", "leaf_size",
    "min_split", "criterion", "max_features", "train_pct", "val_pct", "fit_seconds"])
display(results.round(2))

## Training accuracy and validation accuracy

Training accuracy measures performance on rows used for learning. Validation accuracy measures performance on held-out rows. The gap shows how much better a model performs on familiar examples.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, task in zip(axes, ("image", "sentiment")):
    selected = results[(results.task == task) & (results.criterion == "gini") &
        (((results.model == "tree") & (results.depth == 20 if task == "image" else results.leaf_size == 5) &
          (results.min_split == (20 if task == "image" else 2))) |
         ((results.model == "forest") & (results.trees.isin([30, 60])) & (results.max_features == "sqrt")))]
    labels = ["Tree" if row.model == "tree" else f"Forest {row.trees}" for row in selected.itertuples()]
    x = np.arange(len(selected))
    ax.bar(x - 0.18, selected.train_pct, width=0.36, label="Training")
    ax.bar(x + 0.18, selected.val_pct, width=0.36, label="Validation")
    ax.set(xticks=x, xticklabels=labels, ylabel="Accuracy (%)", title=task.title(), ylim=(0, 100))
    ax.legend()
fig.tight_layout()
plt.show()

Increasing image tree depth from 10 to 20 improved validation accuracy from 7.2% to 9.8%. Increasing the minimum split size to 20 gave 9.9%, a small difference that should not be treated as strong evidence.

On sentiment, a leaf size of 5 slightly improved tree validation accuracy. Increasing the minimum split size to 20 then lowered it. The sentiment forest had a large training–validation gap even after tuning.

The forest scored higher than the single tree on both tasks. Increasing to 60 trees improved image validation accuracy to 16.65%. The sentiment increase was small, from about 57.17% to 57.29%.

## Gini and entropy

These comparisons keep depth and minimum sample settings fixed within each task. Image uses depth 20 and leaf size 1. Sentiment uses depth 10 and leaf size 5.

In [ ]:
criteria = results[(results.model == "tree") & (results.min_split == 2) &
    (((results.task == "image") & (results.depth == 20)) |
     ((results.task == "sentiment") & (results.leaf_size == 5)))]
display(criteria[["task", "criterion", "train_pct", "val_pct", "fit_seconds"]])

Gini was faster and had the higher recorded validation accuracy on both tasks. For images, entropy raised training accuracy substantially without improving validation accuracy. This is evidence of a larger generalization gap, so Gini was kept for the selected models.

## Minimum split size and candidate features

I used the settings from my terminal runs to compare one change at a time. The following rows compare `min_samples_split=2` with 20 for the tree and `max_features="sqrt"` with 0.1 for the 30-tree forest. Image depth stays at 20 with leaf size 1. Sentiment depth stays at 10 with leaf size 5.

In [ ]:
comparison = results[(results.criterion == "gini") &
    (((results.task == "image") & (results.depth == 20) & (results.leaf_size == 1)) |
     ((results.task == "sentiment") & (results.depth == 10) & (results.leaf_size == 5))) &
    ((results.model == "tree") | ((results.model == "forest") & (results.trees == 30)))]
display(comparison[["task", "model", "min_split", "max_features", "train_pct", "val_pct"]].round(2))

The larger minimum split size improved image tree validation accuracy by only 0.1 percentage points and reduced sentiment tree validation accuracy by 0.3 points. I kept 20 for the image tree and 2 for the sentiment tree. Considering 10% of features lowered forest validation accuracy from 15.3% to 14.5% on images and from 57.2% to 56.5% on sentiment. I kept `"sqrt"` for both forests.

## Optional experiment rerun

Set `RUN_EXPERIMENTS = True` in the first code cell and run the notebook again to reproduce every configuration in the recorded table. New measurements are displayed separately so they can be compared with the original results. Runtime measurements depend on the machine.

In [ ]:
if RUN_EXPERIMENTS:
    measured = []
    for row in results.to_dict("records"):
        settings = dict(max_depth=row["depth"], min_samples_leaf=row["leaf_size"],
                        min_samples_split=row["min_split"], criterion=row["criterion"], seed=SEED)
        if row["model"] == "forest":
            features = 0.1 if row["max_features"] == "0.1" else "sqrt"
            model = RandomForestClassifier(n_estimators=row["trees"], max_features=features, **settings)
        else:
            model = DecisionTreeClassifier(**settings)
        print(row["task"], row["model"], settings, flush=True)
        scores, _ = evaluate(model, data[row["task"]])
        measured.append({**row, "train_pct": scores["train_acc"] * 100,
                         "val_pct": scores["val_acc"] * 100, "fit_seconds": scores["fit_time_s"]})
    rerun_results = pd.DataFrame(measured)
    display(rerun_results.round(2))
else:
    print("Experiment rerun is off. The tables show the recorded measurements.")

## Optional error analysis

Set `RUN_ERROR_ANALYSIS = True` to train the selected tree and 60-tree forest on each training split. This section calculates class recall, confusion matrices, frequently confused labels, and the overlap between their errors. The image classes are identified by number because object names were not provided.

In [ ]:
if RUN_ERROR_ANALYSIS:
    for task, depth, leaf in [("image", 20, 1), ("sentiment", 10, 5)]:
        split = data[task]
        models = {
            "tree": DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
                min_samples_split=20 if task == "image" else 2, seed=SEED),
            "forest": RandomForestClassifier(n_estimators=60, max_depth=depth,
                min_samples_leaf=leaf, max_features="sqrt", seed=SEED),
        }
        predictions = {}
        for name, model in models.items():
            print(f"Training {task} {name} for error analysis", flush=True)
            scores, pred = evaluate(model, split)
            predictions[name] = pred
            cm = confusion_matrix(split.y_val, pred, split.n_classes)
            labels = list(range(split.n_classes))
            if task == "sentiment":
                labels = [SENTIMENT_LABELS[i] for i in labels]
            display(pd.DataFrame({"class": labels, "recall": per_class_accuracy(cm)}))
            display(most_confused_pairs(cm, top=5))
            fig, ax = plt.subplots(figsize=(6, 5))
            normalized = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
            image = ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
            ax.set(title=f"{task.title()} {name} validation", xlabel="Predicted class", ylabel="True class")
            if task == "sentiment":
                ax.set_xticks(range(3), labels, rotation=20)
                ax.set_yticks(range(3), labels)
            fig.colorbar(image, ax=ax, label="Fraction of true class")
            fig.tight_layout()
            plt.show()
        tree_ok = predictions["tree"] == split.y_val
        forest_ok = predictions["forest"] == split.y_val
        print("Both wrong:", np.sum(~tree_ok & ~forest_ok))
        print("Forest correct, tree wrong:", np.sum(forest_ok & ~tree_ok))
        print("Tree correct, forest wrong:", np.sum(tree_ok & ~forest_ok))
else:
    print("Error analysis is off. Enable it to calculate findings from validation predictions.")

## Kaggle submissions

The selected forests were retrained on all labeled rows only after choosing their settings. Predictions for the 5,000 image rows and 5,000 sentiment rows were combined by ID using the repository submission helper. The following public scores were reported after upload. They measure a different set of examples from validation.

In [ ]:
display(pd.DataFrame([
    {"submission": "30-tree forest", "image_depth": 20, "sentiment_depth": 10, "public_score": 0.365},
    {"submission": "60-tree forest", "image_depth": 20, "sentiment_depth": 10, "public_score": 0.378},
]))

## What I learned

The random forest scored higher than my single tree on both datasets. Adding trees helped image accuracy more than sentiment accuracy. Both models still struggled on images.

The training score alone was not enough to choose a model. For example, entropy raised image training accuracy to 67.8%, but validation accuracy stayed at 9.7%. I kept Gini because it gave better validation results and trained faster.

One possible reason for the low image scores is that each tree asks about one feature at a time. Useful information may be spread across several features. This is an explanation to investigate alongside the team's k-NN and neural network results.

## Assistance

I used ChatGPT to help with the implementation, experiment setup, explanations, and notebook preparation. The tables show earlier recorded results. The optional experiment cells let these runs be repeated.